# Vesuvius Challenge 2025 - V6 Two-Stage Training

## Key Innovation: Two-Stage Training

The competition is **topology-focused** (TopoScore 30%, VOI 35%, SurfaceDice 35%).
Previous attempts with topology losses (SkeletonRecall, clDice) failed because:
- V2: Introduced topology losses at epoch 300 with high LR → training stopped at Val Dice 0.53
- V4/V5: No topology losses → reached Val Dice 0.68+ but poor topology

**Solution**: Train in two stages to get the best of both worlds.

### Stage 1: High Dice Foundation (epochs 0-600)
- **Loss**: Dice (0.5) + BCE (0.5) only
- **LR**: 0.01 → polynomial decay
- **Goal**: Reach Val Dice 0.65+ (proven achievable)

### Stage 2: Topology Fine-tuning (epochs 600-1200)
- **Loss**: Dice (0.3) + BCE (0.2) + SkeletonRecall (0.25) + clDice (0.25)
- **LR**: 0.001 (10x lower, fresh optimizer)
- **Goal**: Maintain Val Dice 0.60+ while improving topology metrics

### Why This Works
1. Stage 1 builds strong segmentation features without topology constraints
2. Stage 2 starts with much lower LR → gradual adaptation to topology losses
3. This is transfer learning: use high-Dice model as pretrained, fine-tune for topology

In [ ]:
!pip install imagecodecs connected-components-3d tifffile -q

In [ ]:
# =============================================================================
# CELL 1: IMPORTS & CONFIGURATION - V6 TWO-STAGE TRAINING
# =============================================================================
# Optimized for H100 80GB based on:
# "Achieving Sub-7-Minute Epochs: A Blueprint for Optimizing Vesuvius Challenge Training on H100"

import os
import gc
import json
import math
import random
import warnings
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from scipy import ndimage
from scipy.ndimage import distance_transform_edt, gaussian_filter

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION - V6 TWO-STAGE TRAINING (OPTIMIZED FOR H100)
# =============================================================================

@dataclass
class Config:
    # Data paths
    DATA_ROOT: Path = Path("/kaggle/input/3d-volume-training-data")
    CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints_v6")
    LOAD_CHECKPOINT_DIR: Path = Path("/kaggle/working/checkpoints_v6")
    
    # Model architecture
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = None  # [32, 64, 128, 256, 320, 320]
    N_BLOCKS: List[int] = None  # [1, 3, 4, 6, 6, 6]
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    USE_ASPP: bool = True  # Multi-scale context at bottleneck
    
    # ==========================================================================
    # V6 TWO-STAGE TRAINING CONFIGURATION
    # ==========================================================================
    
    # Stage 1: Foundation (epochs 0-600) - Dice + BCE only
    STAGE1_EPOCHS: int = 600
    STAGE1_DICE_WEIGHT: float = 0.5
    STAGE1_BCE_WEIGHT: float = 0.5
    STAGE1_SKELETON_WEIGHT: float = 0.0  # No topology loss
    STAGE1_CLDICE_WEIGHT: float = 0.0    # No topology loss
    STAGE1_LR: float = 0.01
    
    # Stage 2: Topology Fine-tuning (epochs 600-1200)
    STAGE2_EPOCHS: int = 600  # Total = 1200
    STAGE2_DICE_WEIGHT: float = 0.3
    STAGE2_BCE_WEIGHT: float = 0.2
    STAGE2_SKELETON_WEIGHT: float = 0.25  # Add topology
    STAGE2_CLDICE_WEIGHT: float = 0.25    # Add topology
    STAGE2_LR: float = 0.001  # 10x lower
    
    # Topology loss warmup in Stage 2
    SKELETON_WARMUP_EPOCHS: int = 100  # Gradual warmup over 100 epochs
    CLDICE_WARMUP_EPOCHS: int = 100
    
    TOTAL_EPOCHS: int = 1200
    
    # ==========================================================================
    # TRAINING SETTINGS - OPTIMIZED FOR H100 (from blueprint)
    # ==========================================================================
    # Patch 192³ + Batch 4 = ~40GB VRAM usage
    BATCH_SIZE: int = 4
    
    # Increase workers + prefetch to keep GPU fed
    NUM_WORKERS: int = 16        # Increased from 4 (more CPU parallelism)
    PREFETCH_FACTOR: int = 4     # Batches to prefetch per worker
    
    MOMENTUM: float = 0.99
    WEIGHT_DECAY: float = 3e-5
    GRADIENT_CLIP: float = 5.0
    
    # Training control
    RESUME_TRAINING: bool = False
    LOAD_BEST: bool = False
    VALIDATE_EVERY: int = 5
    SAVE_EVERY: int = 10
    
    # Device
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True  # Essential for H100 tensor cores
    
    # Data
    DATA_FRACTION: float = 1.0
    PATCHES_PER_VOLUME: int = 12
    PRELOAD_ALL: bool = True  # Keep all data in RAM (220GB available)
    FG_OVERSAMPLE_RATIO: float = 0.33
    
    # ==========================================================================
    # AUGMENTATION - OPTIMIZED FOR SPEED
    # ==========================================================================
    # Reduced probabilities for slow scipy augmentations
    # This prevents CPU from becoming bottleneck
    
    AUG_ROTATION_RANGE: float = 30.0
    AUG_ROTATION_P: float = 0.1       # Reduced (scipy.ndimage is slow)
    AUG_SCALE_RANGE: Tuple[float, float] = (0.85, 1.15)
    AUG_SCALE_P: float = 0.1          # Reduced
    
    # Fast augmentations (keep these)
    AUG_GAUSSIAN_NOISE_P: float = 0.15
    AUG_GAUSSIAN_NOISE_STD: Tuple[float, float] = (0.0, 0.1)
    AUG_GAUSSIAN_BLUR_P: float = 0.1
    AUG_GAUSSIAN_BLUR_SIGMA: Tuple[float, float] = (0.5, 1.0)
    AUG_BRIGHTNESS_P: float = 0.15
    AUG_BRIGHTNESS_RANGE: float = 0.3
    AUG_CONTRAST_P: float = 0.15
    AUG_CONTRAST_RANGE: Tuple[float, float] = (0.75, 1.25)
    AUG_GAMMA_P: float = 0.15
    AUG_GAMMA_RANGE: Tuple[float, float] = (0.7, 1.5)
    AUG_GAMMA_INVERT_P: float = 0.05
    
    # DISABLED - too slow (scipy.ndimage.zoom)
    AUG_LOW_RES_P: float = 0.0
    AUG_LOW_RES_ZOOM: Tuple[float, float] = (0.5, 1.0)
    
    # Fast - keep at 0.5
    AUG_MIRROR_P: float = 0.5
    
    SEED: int = 42
    
    def __post_init__(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320, 320]
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 3, 4, 6, 6, 6]
        self.CHECKPOINT_DIR = Path(self.CHECKPOINT_DIR)
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        self.LOAD_CHECKPOINT_DIR = Path(self.LOAD_CHECKPOINT_DIR)

cfg = Config()
cfg.__post_init__()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

set_seed(cfg.SEED)

print("="*70)
print("V6 TWO-STAGE TRAINING - OPTIMIZED FOR H100")
print("="*70)
print(f"\n>>> HARDWARE OPTIMIZATION:")
print(f"    BATCH_SIZE: {cfg.BATCH_SIZE} (for 192³ patches)")
print(f"    NUM_WORKERS: {cfg.NUM_WORKERS} (high parallelism)")
print(f"    PREFETCH_FACTOR: {cfg.PREFETCH_FACTOR}")
print(f"    USE_AMP: {cfg.USE_AMP} (H100 tensor cores)")
print(f"\n>>> PATCH SIZE: {cfg.PATCH_SIZE}")
print(f">>> USE_ASPP: {cfg.USE_ASPP}")
print(f"\n>>> AUGMENTATION (speed-optimized):")
print(f"    Rotation/Scale P: {cfg.AUG_ROTATION_P}/{cfg.AUG_SCALE_P}")
print(f"    Low-res P: {cfg.AUG_LOW_RES_P} (disabled)")
print(f"    Mirror P: {cfg.AUG_MIRROR_P}")
print(f"\n>>> STAGE 1 (epochs 0-{cfg.STAGE1_EPOCHS}): Foundation")
print(f"    Loss: Dice={cfg.STAGE1_DICE_WEIGHT}, BCE={cfg.STAGE1_BCE_WEIGHT}")
print(f"    LR: {cfg.STAGE1_LR}")
print(f"\n>>> STAGE 2 (epochs {cfg.STAGE1_EPOCHS}-{cfg.TOTAL_EPOCHS}): Topology")
print(f"    Loss: Dice={cfg.STAGE2_DICE_WEIGHT}, BCE={cfg.STAGE2_BCE_WEIGHT}, Skel={cfg.STAGE2_SKELETON_WEIGHT}, clDice={cfg.STAGE2_CLDICE_WEIGHT}")
print(f"    LR: {cfg.STAGE2_LR}")
print(f"\n>>> Total epochs: {cfg.TOTAL_EPOCHS}")
print("="*70)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# =============================================================================
# CELL 2: CHECKPOINT MANAGER (same as V4)
# =============================================================================

class CheckpointManager:
    def __init__(self, save_dir: Path, load_dir: Path = None, max_keep: int = 5):
        self.save_dir = Path(save_dir)
        self.load_dir = Path(load_dir) if load_dir else self.save_dir
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.max_keep = max_keep
        self.best_score = -1
        self.best_epoch = -1
        self.history = []
    
    def save(self, model, optimizer, scheduler, scaler, epoch: int, metrics: dict):
        model_to_save = model._orig_mod if hasattr(model, '_orig_mod') else model
        
        ckpt = {
            'epoch': epoch,
            'model_state_dict': model_to_save.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'metrics': metrics,
            'best_score': self.best_score,
            'best_epoch': self.best_epoch,
            'config': {
                'features': cfg.FEATURES,
                'n_blocks': cfg.N_BLOCKS,
                'patch_size': cfg.PATCH_SIZE,
                'version': 'V6',
            }
        }
        
        torch.save(ckpt, self.save_dir / 'last_checkpoint.pth')
        
        val_dice = metrics.get('val_dice', 0)
        if val_dice > 0 and val_dice > self.best_score:
            self.best_score = val_dice
            self.best_epoch = epoch
            ckpt['best_score'] = self.best_score
            ckpt['best_epoch'] = self.best_epoch
            torch.save(ckpt, self.save_dir / 'best_model.pth')
            print(f"  >>> New best model! Val Dice: {val_dice:.4f}")
        
        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            torch.save(ckpt, self.save_dir / f'checkpoint_epoch_{epoch+1}.pth')
            self._cleanup_old_checkpoints()
        
        self.history.append({'epoch': epoch, **metrics})
        with open(self.save_dir / 'history.json', 'w') as f:
            json.dump(self.history, f, indent=2)
    
    def load(self, model, optimizer=None, scheduler=None, scaler=None,
             checkpoint_path=None, load_best: bool = False) -> int:
        if checkpoint_path is None:
            if load_best:
                checkpoint_path = self.load_dir / 'best_model.pth'
            else:
                checkpoint_path = self.load_dir / 'last_checkpoint.pth'
        
        checkpoint_path = Path(checkpoint_path)
        if not checkpoint_path.exists():
            print("No checkpoint found, starting fresh training")
            return 0
        
        print(f"Loading checkpoint from {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=cfg.DEVICE, weights_only=False)
        
        model_to_load = model._orig_mod if hasattr(model, '_orig_mod') else model
        model_to_load.load_state_dict(ckpt['model_state_dict'], strict=True)
        
        if optimizer and 'optimizer_state_dict' in ckpt:
            try:
                optimizer.load_state_dict(ckpt['optimizer_state_dict'])
                print("  Optimizer state loaded")
            except:
                print("  Warning: Could not load optimizer state")
        
        if scaler and 'scaler_state_dict' in ckpt and ckpt['scaler_state_dict']:
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        
        self.best_score = ckpt.get('best_score', -1)
        self.best_epoch = ckpt.get('best_epoch', -1)
        
        history_path = self.save_dir / 'history.json'
        if history_path.exists():
            with open(history_path, 'r') as f:
                self.history = json.load(f)
        
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {ckpt['epoch']}")
        if self.best_score > 0:
            print(f"Best score so far: {self.best_score:.4f} at epoch {self.best_epoch}")
        return start_epoch
    
    def _cleanup_old_checkpoints(self):
        checkpoints = sorted(self.save_dir.glob('checkpoint_epoch_*.pth'))
        while len(checkpoints) > self.max_keep:
            checkpoints[0].unlink()
            checkpoints = checkpoints[1:]

print("CheckpointManager defined")

In [ ]:
# =============================================================================
# CELL 3: HELPER FUNCTIONS
# =============================================================================

def load_volume(path):
    """Load volume from TIFF file."""
    return tifffile.imread(str(path))


print("Helper functions defined")

In [ ]:
# =============================================================================
# CELL 4: DATASET WITH FULL nnUNet AUGMENTATION (TIFF format)
# =============================================================================

def augment_spatial(image, label, cfg):
    """Rotation and scaling augmentation."""
    do_rotation = random.random() < cfg.AUG_ROTATION_P
    do_scale = random.random() < cfg.AUG_SCALE_P

    if not do_rotation and not do_scale:
        return image, label

    center = np.array(image.shape) / 2.0
    rot_range = cfg.AUG_ROTATION_RANGE * np.pi / 180.0

    if do_rotation:
        angle = random.uniform(-rot_range, rot_range)
        axis_pair = random.choice([(0, 1), (0, 2), (1, 2)])
    else:
        angle = 0
        axis_pair = (0, 1)

    if do_scale:
        scale = random.uniform(*cfg.AUG_SCALE_RANGE)
    else:
        scale = 1.0

    R = np.eye(3)
    c, s = np.cos(angle), np.sin(angle)
    i, j = axis_pair
    R[i, i] = c
    R[i, j] = -s
    R[j, i] = s
    R[j, j] = c
    R = R / scale
    offset = center - R @ center

    image_out = ndimage.affine_transform(image, R, offset=offset, order=3, mode='reflect')
    label_out = ndimage.affine_transform(
        label.astype(np.float32), R, offset=offset, order=0, mode='constant', cval=2
    ).astype(np.uint8)

    return image_out, label_out


def augment_gaussian_noise(image, cfg):
    if random.random() >= cfg.AUG_GAUSSIAN_NOISE_P:
        return image
    std = random.uniform(*cfg.AUG_GAUSSIAN_NOISE_STD)
    noise = np.random.normal(0, std, image.shape).astype(np.float32)
    return image + noise


def augment_gaussian_blur(image, cfg):
    if random.random() >= cfg.AUG_GAUSSIAN_BLUR_P:
        return image
    sigma = random.uniform(*cfg.AUG_GAUSSIAN_BLUR_SIGMA)
    return gaussian_filter(image, sigma=sigma).astype(np.float32)


def augment_brightness(image, cfg):
    if random.random() >= cfg.AUG_BRIGHTNESS_P:
        return image
    offset = random.uniform(-cfg.AUG_BRIGHTNESS_RANGE, cfg.AUG_BRIGHTNESS_RANGE)
    return image + offset


def augment_contrast(image, cfg):
    if random.random() >= cfg.AUG_CONTRAST_P:
        return image
    factor = random.uniform(*cfg.AUG_CONTRAST_RANGE)
    mean = image.mean()
    return (image - mean) * factor + mean


def augment_gamma(image, cfg):
    if random.random() < cfg.AUG_GAMMA_P:
        gamma = random.uniform(*cfg.AUG_GAMMA_RANGE)
        mn, mx = image.min(), image.max()
        rng = mx - mn + 1e-8
        image = ((image - mn) / rng) ** gamma * rng + mn

    if random.random() < cfg.AUG_GAMMA_INVERT_P:
        gamma = random.uniform(*cfg.AUG_GAMMA_RANGE)
        mn, mx = image.min(), image.max()
        rng = mx - mn + 1e-8
        image = rng - ((rng - (image - mn)) / rng) ** gamma * rng + mn

    return image


def augment_simulate_low_resolution(image, cfg):
    if random.random() >= cfg.AUG_LOW_RES_P:
        return image

    axis = random.randint(0, 2)
    zoom_factor = random.uniform(*cfg.AUG_LOW_RES_ZOOM)

    if zoom_factor >= 0.99:
        return image

    shape = list(image.shape)
    zoom = [1.0, 1.0, 1.0]
    zoom[axis] = zoom_factor

    downsampled = ndimage.zoom(image, zoom, order=0)
    zoom_back = [1.0, 1.0, 1.0]
    zoom_back[axis] = shape[axis] / downsampled.shape[axis]
    upsampled = ndimage.zoom(downsampled, zoom_back, order=3)

    if upsampled.shape != tuple(shape):
        result = np.zeros(shape, dtype=np.float32)
        slices = tuple(slice(0, min(s, u)) for s, u in zip(shape, upsampled.shape))
        result[slices] = upsampled[slices]
        return result

    return upsampled.astype(np.float32)


def augment_mirror(image, label, cfg):
    for axis in range(3):
        if random.random() < cfg.AUG_MIRROR_P:
            image = np.flip(image, axis)
            label = np.flip(label, axis)
    return np.ascontiguousarray(image), np.ascontiguousarray(label)


def apply_nnunet_augmentations(image, label, cfg):
    """Full nnUNet augmentation pipeline."""
    image, label = augment_spatial(image, label, cfg)
    image = augment_gaussian_noise(image, cfg)
    image = augment_gaussian_blur(image, cfg)
    image = augment_brightness(image, cfg)
    image = augment_contrast(image, cfg)
    image = augment_gamma(image, cfg)
    image = augment_simulate_low_resolution(image, cfg)
    image, label = augment_mirror(image, label, cfg)
    return image.astype(np.float32), label


class VesuviusDatasetV6(Dataset):
    """
    V6 Dataset with:
    - TIFF file loading
    - Full nnUNet augmentation
    - Foreground oversampling

    Label handling:
    - Class 0 = background
    - Class 1 = foreground
    - Class 2 = ignore mask (excluded from loss)
    """
    def __init__(
        self,
        csv_path: Path,
        images_dir: Path,
        labels_dir: Path,
        patch_size: Tuple[int, int, int] = (192, 192, 192),
        is_train: bool = True,
        data_fraction: float = 1.0,
        patches_per_volume: int = 8,
        preload: bool = True,
    ):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume

        df = pd.read_csv(csv_path)

        # Check for TIFF files
        valid_ids = []
        for idx in df['id'].values:
            img_path = self.images_dir / f"{idx}.tif"
            lbl_path = self.labels_dir / f"{idx}.tif"
            if img_path.exists() and lbl_path.exists():
                valid_ids.append(idx)

        print(f"Found {len(valid_ids)} volumes (TIFF format)")

        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.shuffle(valid_ids)
            valid_ids = valid_ids[:n]

        self.volume_ids = valid_ids
        self.volumes = {}
        self.fg_coords = {}

        if preload:
            print(f"Preloading {len(self.volume_ids)} volumes...")
            load_start = time.time()

            for vol_id in tqdm(self.volume_ids, desc="Loading"):
                # Load TIFF files
                img = load_volume(self.images_dir / f"{vol_id}.tif").astype(np.float32)
                lbl = load_volume(self.labels_dir / f"{vol_id}.tif").astype(np.uint8)

                # Normalize image (z-score normalization)
                img = (img - img.mean()) / (img.std() + 1e-8)

                self.volumes[vol_id] = (img, lbl)

                # Store FG coordinates for oversampling
                fg = np.argwhere(lbl == 1)
                if len(fg) > 10000:
                    fg = fg[np.random.choice(len(fg), 10000, replace=False)]
                self.fg_coords[vol_id] = fg if len(fg) > 0 else None

            load_time = time.time() - load_start
            sample_vol = next(iter(self.volumes.values()))
            vol_size_mb = (sample_vol[0].nbytes + sample_vol[1].nbytes) / 1e6
            total_gb = vol_size_mb * len(self.volumes) / 1000
            print(f"Loaded {len(self.volumes)} volumes ({total_gb:.1f} GB) in {load_time:.1f}s")
            print(f"  Average: {load_time/len(self.volumes):.2f}s per volume")

        print(f"Dataset ready: {len(self)} samples")

    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume

    def __getitem__(self, idx):
        vol_idx = idx // self.patches_per_volume
        vol_id = self.volume_ids[vol_idx]

        image, label = self.volumes[vol_id]
        d, h, w = image.shape
        pd, ph, pw = self.patch_size

        # Pad if needed
        if d < pd or h < ph or w < pw:
            pad_d = max(0, pd - d)
            pad_h = max(0, ph - h)
            pad_w = max(0, pw - w)
            image = np.pad(image, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='reflect')
            label = np.pad(label, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='constant', constant_values=2)
            d, h, w = image.shape

        # Foreground oversampling
        fg = self.fg_coords.get(vol_id)
        force_fg = self.is_train and random.random() < cfg.FG_OVERSAMPLE_RATIO and fg is not None and len(fg) > 0

        if force_fg:
            c = fg[random.randint(0, len(fg) - 1)]
            z = max(0, min(c[0] - pd // 2, d - pd))
            y = max(0, min(c[1] - ph // 2, h - ph))
            x = max(0, min(c[2] - pw // 2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))

        img_patch = image[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_patch = label[z:z+pd, y:y+ph, x:x+pw].copy()

        # Apply augmentation (during training)
        if self.is_train:
            img_patch, lbl_patch = apply_nnunet_augmentations(img_patch, lbl_patch, cfg)

        # Simple label handling
        target = (lbl_patch == 1).astype(np.float32)
        ignore_mask = (lbl_patch == 2).astype(np.float32)

        return {
            'image': torch.from_numpy(img_patch).unsqueeze(0).float(),
            'label': torch.from_numpy(target).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ignore_mask).unsqueeze(0).float(),
        }


print("V6 Dataset (TIFF format) with augmentation ready")

In [ ]:
# =============================================================================
# CELL 5: MODEL WITH ASPP (Atrous Spatial Pyramid Pooling)
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True):
        super(SafeInstanceNorm3d, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        
        if self.affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)
    
    def forward(self, x):
        mean = x.mean(dim=[2, 3, 4], keepdim=True)
        var = x.var(dim=[2, 3, 4], keepdim=True, unbiased=False)
        var_safe = torch.clamp(var, min=self.eps)
        x_norm = (x - mean) / torch.sqrt(var_safe + self.eps)
        
        if self.affine:
            x_norm = self.weight.view(1, -1, 1, 1, 1) * x_norm + self.bias.view(1, -1, 1, 1, 1)
        
        return x_norm


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=True),
            SafeInstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ConvBlock(channels, channels) for _ in range(n_convs)]
        )
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


# =============================================================================
# ASPP (Atrous Spatial Pyramid Pooling) for 3D
# =============================================================================

class ASPP3D(nn.Module):
    """
    Atrous Spatial Pyramid Pooling for 3D volumes.
    Captures multi-scale context at the bottleneck without increasing depth.
    
    With 6³ bottleneck resolution and dilations [1, 2, 4, 8]:
    - rate=1: 3x3x3 receptive field (local details)
    - rate=2: 5x5x5 receptive field
    - rate=4: 9x9x9 receptive field  
    - rate=8: 17x17x17 receptive field (covers ~3x bottleneck)
    
    Plus Global Average Pooling for image-level features.
    """
    def __init__(self, in_channels, out_channels, dilations=[1, 2, 4, 8]):
        super().__init__()
        
        # 1x1 convolution branch (replaces rate=1 dilated conv)
        self.conv1x1 = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, 1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.01, inplace=True),
        )
        
        # Dilated convolution branches (rates 2, 4, 8)
        self.dilated_convs = nn.ModuleList()
        for d in dilations[1:]:  # Skip rate=1
            self.dilated_convs.append(nn.Sequential(
                nn.Conv3d(in_channels, out_channels, 3, padding=d, dilation=d, bias=False),
                SafeInstanceNorm3d(out_channels),
                nn.LeakyReLU(0.01, inplace=True),
            ))
        
        # Global Average Pooling branch (image-level features)
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Conv3d(in_channels, out_channels, 1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.01, inplace=True),
        )
        
        # Fusion: 1x1 + dilated branches + GAP = len(dilations) + 1 total
        n_branches = len(dilations) + 1  # 1 (1x1) + 3 (dilated) + 1 (GAP) = 5
        self.project = nn.Sequential(
            nn.Conv3d(out_channels * n_branches, out_channels, 1, bias=False),
            SafeInstanceNorm3d(out_channels),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Dropout3d(0.1),
        )
    
    def forward(self, x):
        size = x.shape[2:]
        
        # 1x1 branch
        out1x1 = self.conv1x1(x)
        
        # Dilated branches
        dilated_outs = [conv(x) for conv in self.dilated_convs]
        
        # GAP branch (upsample back to input size)
        gap_out = self.gap(x)
        gap_out = F.interpolate(gap_out, size=size, mode='trilinear', align_corners=False)
        
        # Concatenate all branches
        out = torch.cat([out1x1] + dilated_outs + [gap_out], dim=1)
        
        # Project back to out_channels
        return self.project(out)


# =============================================================================
# ResEncUNet3D with optional ASPP
# =============================================================================

class ResEncUNet3D(nn.Module):
    def __init__(
        self,
        in_ch: int = 1,
        out_ch: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_deep_supervision: bool = True,
        use_aspp: bool = True,  # NEW: Add ASPP at bottleneck
    ):
        super().__init__()
        
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.use_aspp = use_aspp
        self.n_stages = len(features)
        
        # Encoders
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat, n_convs=2) for _ in range(nb)]
            )
            self.encoders.append(encoder)
            
            if use_scse:
                self.attentions.append(scSEBlock(feat))
            else:
                self.attentions.append(nn.Identity())
            
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2, bias=True))
        
        # ASPP at bottleneck for multi-scale context
        if use_aspp:
            self.aspp = ASPP3D(features[-1], features[-1], dilations=[1, 2, 4, 8])
        
        # Decoders
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2, bias=True))
            self.dec_convs.append(ConvBlock(out_feat * 2, out_feat))
        
        self.final = nn.Conv3d(features[0], out_ch, 1, bias=True)
        
        # Deep supervision heads
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            n_ds_outputs = min(4, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1, bias=True))
    
    def forward(self, x):
        enc_features = []
        
        # Encoder path
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        
        # Apply ASPP at bottleneck for multi-scale context
        if self.use_aspp:
            x = self.aspp(x)
        
        ds_outputs = []
        
        # Decoder path
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final(x)
        
        if self.use_deep_supervision and self.training:
            return {'output': out, 'deep': ds_outputs}
        return out


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("ResEncUNet3D with ASPP defined")
print(f"  - USE_ASPP: Adds multi-scale context at bottleneck")
print(f"  - Dilation rates: [1, 2, 4, 8] + Global Average Pooling")

In [ ]:
# =============================================================================
# CELL 6: V6 LOSS FUNCTIONS - TWO-STAGE WITH TOPOLOGY LOSSES
# =============================================================================

class DiceLoss(nn.Module):
    """Standard Dice Loss."""
    def __init__(self, smooth: float = 1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        batch_size = pred.shape[0]
        dice_sum = 0.0
        for b in range(batch_size):
            p = pred[b].reshape(-1)
            t = target[b].reshape(-1)
            intersection = (p * t).sum()
            dice_sum += (2 * intersection + self.smooth) / (p.sum() + t.sum() + self.smooth)
        
        return 1 - dice_sum / batch_size


class CEWithMask(nn.Module):
    """Binary Cross-Entropy with ignore mask."""
    def forward(self, pred, target, mask=None):
        if mask is not None:
            valid = 1 - mask
            loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
            loss = (loss * valid).sum() / (valid.sum() + 1e-8)
        else:
            loss = F.binary_cross_entropy_with_logits(pred, target)
        return loss


def soft_skeletonize(x, num_iter=10):
    """
    Soft skeletonization using iterative min-max pooling.
    From clDice paper (CVPR 2021).
    """
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    return x


class SkeletonRecallLoss(nn.Module):
    """
    Skeleton Recall Loss from Skeleton Recall Loss (ECCV 2024).
    Preserves connectivity by ensuring predictions cover skeleton.
    """
    def __init__(self, smooth: float = 1e-5, tube_radius: int = 2, num_iter: int = 10):
        super().__init__()
        self.smooth = smooth
        self.tube_radius = tube_radius
        self.num_iter = num_iter
    
    def forward(self, pred, target, mask=None):
        pred_sigmoid = torch.sigmoid(pred)
        
        if mask is not None:
            pred_sigmoid = pred_sigmoid * (1 - mask)
            target = target * (1 - mask)
        
        # Soft skeletonization
        target_skel = soft_skeletonize(target, num_iter=self.num_iter)
        
        # Tube dilation
        if self.tube_radius > 0:
            target_tube = F.max_pool3d(
                target_skel,
                kernel_size=2 * self.tube_radius + 1,
                stride=1,
                padding=self.tube_radius
            )
        else:
            target_tube = target_skel
        
        # Clamp to prevent extreme values
        target_tube = torch.clamp(target_tube, 0, 1)
        
        recall = (pred_sigmoid * target_tube).sum() / (target_tube.sum() + self.smooth)
        recall = torch.clamp(recall, 0, 1)
        
        return 1 - recall


class SoftClDiceLoss(nn.Module):
    """
    Soft clDice (centerline Dice) loss from CVPR 2021.
    Preserves topology by focusing on skeleton connectivity.
    """
    def __init__(self, smooth: float = 1e-5, num_iter: int = 10):
        super().__init__()
        self.smooth = smooth
        self.num_iter = num_iter
    
    def forward(self, pred, target, mask=None):
        pred_sigmoid = torch.sigmoid(pred)
        
        if mask is not None:
            pred_sigmoid = pred_sigmoid * (1 - mask)
            target = target * (1 - mask)
        
        # Soft skeletonization
        skel_pred = soft_skeletonize(pred_sigmoid, self.num_iter)
        skel_target = soft_skeletonize(target, self.num_iter)
        
        # Topology precision and sensitivity
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_target * pred_sigmoid).sum() + self.smooth) / (skel_target.sum() + self.smooth)
        
        cldice = 2 * tprec * tsens / (tprec + tsens + self.smooth)
        cldice = torch.clamp(cldice, 0, 1)
        
        return 1 - cldice


class TwoStageCombinedLoss(nn.Module):
    """
    Two-Stage Combined Loss for V6:
    - Stage 1 (0-600): Dice + BCE only
    - Stage 2 (600-1200): Dice + BCE + SkeletonRecall + clDice
    
    Key innovation: Topology losses are added in Stage 2 with gradual warmup.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.stage1_epochs = cfg.STAGE1_EPOCHS
        
        # Stage 1 weights
        self.s1_dice = cfg.STAGE1_DICE_WEIGHT
        self.s1_bce = cfg.STAGE1_BCE_WEIGHT
        
        # Stage 2 weights
        self.s2_dice = cfg.STAGE2_DICE_WEIGHT
        self.s2_bce = cfg.STAGE2_BCE_WEIGHT
        self.s2_skel = cfg.STAGE2_SKELETON_WEIGHT
        self.s2_cldice = cfg.STAGE2_CLDICE_WEIGHT
        
        self.skel_warmup = cfg.SKELETON_WARMUP_EPOCHS
        self.cldice_warmup = cfg.CLDICE_WARMUP_EPOCHS
        
        # Deep supervision weights
        raw = [1.0 / (2 ** i) for i in range(4)]
        self.ds_weights = [w / sum(raw) for w in raw]
        
        # Loss functions
        self.dice_loss = DiceLoss()
        self.bce_loss = CEWithMask()
        self.skeleton_loss = SkeletonRecallLoss(num_iter=10)
        self.cldice_loss = SoftClDiceLoss(num_iter=10)
    
    def _warmup_scale(self, epoch, warmup):
        """Gradual warmup in first `warmup` epochs of Stage 2."""
        stage2_epoch = epoch - self.stage1_epochs
        if stage2_epoch < 0:
            return 0.0
        if stage2_epoch < warmup:
            return stage2_epoch / warmup
        return 1.0
    
    def forward(self, output, target, ignore_mask, epoch):
        if isinstance(output, dict):
            pred = output['output']
            deep_outputs = output.get('deep', [])
        else:
            pred = output
            deep_outputs = []
        
        # Determine stage
        in_stage1 = epoch < self.stage1_epochs
        
        # Compute base losses
        dice = self.dice_loss(pred, target, ignore_mask)
        bce = self.bce_loss(pred, target, ignore_mask)
        
        if in_stage1:
            # Stage 1: Dice + BCE only
            total = self.s1_dice * dice + self.s1_bce * bce
            skel_val = 0.0
            cldice_val = 0.0
            skel_scale = 0.0
            cldice_scale = 0.0
        else:
            # Stage 2: Add topology losses with warmup
            skel_scale = self._warmup_scale(epoch, self.skel_warmup)
            cldice_scale = self._warmup_scale(epoch, self.cldice_warmup)
            
            if skel_scale > 0:
                skel = self.skeleton_loss(pred, target, ignore_mask)
                skel = torch.clamp(skel, 0, 2)  # Prevent explosion
            else:
                skel = torch.tensor(0.0, device=pred.device)
            
            if cldice_scale > 0:
                cldice = self.cldice_loss(pred, target, ignore_mask)
                cldice = torch.clamp(cldice, 0, 2)
            else:
                cldice = torch.tensor(0.0, device=pred.device)
            
            total = (
                self.s2_dice * dice +
                self.s2_bce * bce +
                self.s2_skel * skel_scale * skel +
                self.s2_cldice * cldice_scale * cldice
            )
            skel_val = skel.item() if skel_scale > 0 else 0.0
            cldice_val = cldice.item() if cldice_scale > 0 else 0.0
        
        # Deep supervision (same for both stages)
        for i, ds_out in enumerate(deep_outputs):
            if i >= len(self.ds_weights):
                break
            ds_target = F.interpolate(target, size=ds_out.shape[2:], mode='nearest')
            ds_mask = F.interpolate(ignore_mask, size=ds_out.shape[2:], mode='nearest')
            ds_dice = self.dice_loss(ds_out, ds_target, ds_mask)
            ds_bce = self.bce_loss(ds_out, ds_target, ds_mask)
            
            if in_stage1:
                ds_loss = self.s1_dice * ds_dice + self.s1_bce * ds_bce
            else:
                ds_loss = self.s2_dice * ds_dice + self.s2_bce * ds_bce
            total = total + self.ds_weights[i] * ds_loss
        
        total = torch.clamp(total, 0, 10)
        
        return {
            'total': total,
            'dice': dice.item(),
            'bce': bce.item(),
            'skeleton': skel_val,
            'cldice': cldice_val,
            'skel_scale': skel_scale if not in_stage1 else 0.0,
            'cldice_scale': cldice_scale if not in_stage1 else 0.0,
            'stage': 1 if in_stage1 else 2,
        }


print("V6 Two-Stage Loss Functions defined")
print(f"\nStage 1 (epochs 0-{cfg.STAGE1_EPOCHS}):")
print(f"  Dice: {cfg.STAGE1_DICE_WEIGHT}, BCE: {cfg.STAGE1_BCE_WEIGHT}")
print(f"\nStage 2 (epochs {cfg.STAGE1_EPOCHS}-{cfg.TOTAL_EPOCHS}):")
print(f"  Dice: {cfg.STAGE2_DICE_WEIGHT}, BCE: {cfg.STAGE2_BCE_WEIGHT}")
print(f"  SkeletonRecall: {cfg.STAGE2_SKELETON_WEIGHT}, clDice: {cfg.STAGE2_CLDICE_WEIGHT}")
print(f"  Warmup: {cfg.SKELETON_WARMUP_EPOCHS} epochs")

In [ ]:
# =============================================================================
# CELL 7: TRAINING FUNCTIONS - TWO-STAGE AWARE
# =============================================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    """Train one epoch with two-stage loss tracking."""
    model.train()
    
    total_loss = 0
    total_dice = 0
    total_bce = 0
    total_skel = 0
    total_cldice = 0
    n_batches = 0
    nan_batches = 0
    current_stage = 1
    
    pbar = tqdm(total=len(loader), desc=f"Epoch {epoch+1}",
                file=sys.stdout, dynamic_ncols=True)
    
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        ignore = batch['ignore_mask'].to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            losses = criterion(output, labels, ignore, epoch)
        
        if torch.isnan(losses['total']) or torch.isinf(losses['total']):
            nan_batches += 1
            if nan_batches <= 3:
                print(f"\n>>> NaN loss at batch {n_batches}! Skipping...")
            pbar.update(1)
            continue
        
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
        
        if torch.isnan(grad_norm) or torch.isinf(grad_norm):
            nan_batches += 1
            scaler.update()
            pbar.update(1)
            continue
        
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += losses['total'].item()
        total_dice += losses['dice']
        total_bce += losses['bce']
        total_skel += losses.get('skeleton', 0)
        total_cldice += losses.get('cldice', 0)
        current_stage = losses.get('stage', 1)
        n_batches += 1
        
        pbar.update(1)
        
        # Update progress bar based on stage
        if current_stage == 1:
            pbar.set_postfix({
                'S': 1,
                'loss': f"{losses['total'].item():.3f}",
                'dice': f"{losses['dice']:.3f}",
            })
        else:
            pbar.set_postfix({
                'S': 2,
                'loss': f"{losses['total'].item():.3f}",
                'dice': f"{losses['dice']:.3f}",
                'skel': f"{losses.get('skeleton', 0):.3f}",
            })
    
    pbar.close()
    
    if nan_batches > 0:
        print(f"\n>>> WARNING: {nan_batches} NaN batches skipped!")
    
    if n_batches == 0:
        return None
    
    return {
        'train_loss': total_loss / n_batches,
        'train_dice_loss': total_dice / n_batches,
        'train_bce_loss': total_bce / n_batches,
        'train_skel_loss': total_skel / n_batches,
        'train_cldice_loss': total_cldice / n_batches,
        'stage': current_stage,
        'nan_batches': nan_batches,
    }


@torch.no_grad()
def validate(model, loader, device):
    """Validation with per-sample Dice."""
    model.eval()
    
    total_dice = 0
    n_samples = 0
    
    for batch in tqdm(loader, desc="Val", file=sys.stdout, leave=False):
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].numpy()
        ignore = batch['ignore_mask'].numpy()
        
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            if isinstance(output, dict):
                output = output['output']
            probs = torch.sigmoid(output).cpu().numpy()
        
        for b in range(images.shape[0]):
            pred = (probs[b, 0] > 0.5).astype(np.uint8)
            tgt = labels[b, 0].astype(np.uint8)
            ign = ignore[b, 0] > 0.5
            pred[ign] = 0
            tgt[ign] = 0
            
            inter = (pred & tgt).sum()
            union = pred.sum() + tgt.sum()
            total_dice += (2 * inter + 1e-5) / (union + 1e-5)
            n_samples += 1
    
    return {'val_dice': total_dice / max(n_samples, 1)}


print("Training functions with two-stage tracking ready")

In [ ]:
# =============================================================================
# CELL 8: MAIN TRAINING LOOP WITH STAGE TRANSITION (H100 OPTIMIZED)
# =============================================================================

def main_training():
    """V6 Two-Stage Training: Foundation → Topology Fine-tuning."""
    
    print("="*70)
    print("VESUVIUS V6 TWO-STAGE TRAINING - H100 OPTIMIZED")
    print("="*70)
    print(f"\n>>> Hardware settings:")
    print(f"    BATCH_SIZE: {cfg.BATCH_SIZE}")
    print(f"    NUM_WORKERS: {cfg.NUM_WORKERS}")
    print(f"    PREFETCH_FACTOR: {cfg.PREFETCH_FACTOR}")
    print(f"    USE_AMP: {cfg.USE_AMP}")
    print(f"\n>>> Stage 1 (epochs 0-{cfg.STAGE1_EPOCHS}): Dice + BCE")
    print(f">>> Stage 2 (epochs {cfg.STAGE1_EPOCHS}-{cfg.TOTAL_EPOCHS}): + SkeletonRecall + clDice")
    print(f">>> ASPP: {cfg.USE_ASPP} (multi-scale context at bottleneck)")
    print("="*70)
    
    train_csv = cfg.DATA_ROOT / "train.csv"
    train_images = cfg.DATA_ROOT / "train_images"
    train_labels = cfg.DATA_ROOT / "train_labels"
    
    print("\n[1/4] Loading training data...")
    train_dataset = VesuviusDatasetV6(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=True,
        data_fraction=cfg.DATA_FRACTION,
        patches_per_volume=cfg.PATCHES_PER_VOLUME,
        preload=True,
    )
    
    print("\n[2/4] Loading validation data...")
    val_dataset = VesuviusDatasetV6(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=False,
        data_fraction=0.15,
        patches_per_volume=1,
        preload=True,
    )
    
    # H100 optimized DataLoader settings
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True,
        prefetch_factor=cfg.PREFETCH_FACTOR,  # From config
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )
    
    print(f"\nTrain: {len(train_dataset)} samples, {len(train_loader)} batches/epoch")
    print(f"Val: {len(val_dataset)} samples")
    
    print("\n[3/4] Creating model...")
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
        use_aspp=cfg.USE_ASPP,
    ).to(cfg.DEVICE)
    
    # torch.compile for H100 optimization
    if hasattr(torch, 'compile'):
        print("Compiling model with torch.compile()...")
        model = torch.compile(model, mode='reduce-overhead')
    
    n_params = count_parameters(model)
    print(f"Parameters: {n_params:,}")
    if cfg.USE_ASPP:
        print(f"  (includes ASPP for multi-scale context)")
    
    # Two-Stage Loss
    criterion = TwoStageCombinedLoss(cfg)
    
    # Initial optimizer for Stage 1
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=cfg.STAGE1_LR,
        momentum=cfg.MOMENTUM,
        weight_decay=cfg.WEIGHT_DECAY,
        nesterov=True,
    )
    
    # Stage 1 scheduler (polynomial decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda e: (1 - e / cfg.STAGE1_EPOCHS) ** 0.9
    )
    
    scaler = GradScaler(enabled=cfg.USE_AMP)
    
    ckpt_mgr = CheckpointManager(
        save_dir=cfg.CHECKPOINT_DIR,
        load_dir=cfg.LOAD_CHECKPOINT_DIR,
    )
    
    print("\n[4/4] Checking for checkpoint...")
    if cfg.RESUME_TRAINING:
        start_epoch = ckpt_mgr.load(model, optimizer, None, scaler, load_best=cfg.LOAD_BEST)
        # If resuming in Stage 2, recreate scheduler
        if start_epoch >= cfg.STAGE1_EPOCHS:
            optimizer = torch.optim.SGD(
                model.parameters(),
                lr=cfg.STAGE2_LR,
                momentum=cfg.MOMENTUM,
                weight_decay=cfg.WEIGHT_DECAY,
                nesterov=True,
            )
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=cfg.STAGE2_EPOCHS,
                eta_min=1e-6,
            )
            for _ in range(start_epoch - cfg.STAGE1_EPOCHS):
                scheduler.step()
            print(f"  Resumed in Stage 2, LR: {optimizer.param_groups[0]['lr']:.6f}")
    else:
        start_epoch = 0
        print("Starting fresh training from epoch 0 (Stage 1)")
    
    # Track stage transition
    stage2_initialized = start_epoch >= cfg.STAGE1_EPOCHS
    
    print(f"\nStarting from epoch {start_epoch + 1}")
    print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")
    print("="*70)
    print("TRAINING STARTED")
    print("="*70)
    
    for epoch in range(start_epoch, cfg.TOTAL_EPOCHS):
        epoch_start = time.time()
        
        # =======================================================
        # STAGE TRANSITION AT EPOCH 600
        # =======================================================
        if epoch == cfg.STAGE1_EPOCHS and not stage2_initialized:
            print("\n" + "="*70)
            print(">>> ENTERING STAGE 2: TOPOLOGY FINE-TUNING")
            print("="*70)
            
            optimizer = torch.optim.SGD(
                model.parameters(),
                lr=cfg.STAGE2_LR,
                momentum=cfg.MOMENTUM,
                weight_decay=cfg.WEIGHT_DECAY,
                nesterov=True,
            )
            
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=cfg.STAGE2_EPOCHS,
                eta_min=1e-6,
            )
            
            scaler = GradScaler(enabled=cfg.USE_AMP)
            
            stage2_initialized = True
            print(f">>> LR reset: {cfg.STAGE1_LR} → {cfg.STAGE2_LR} (10x lower)")
            print(f">>> Adding SkeletonRecall + clDice losses with {cfg.SKELETON_WARMUP_EPOCHS}-epoch warmup")
            print("="*70 + "\n")
        # =======================================================
        
        # Train one epoch
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, cfg.DEVICE, epoch
        )
        
        if train_metrics is None:
            print("\n>>> Training failed!")
            break
        
        scheduler.step()
        
        # Validation
        if (epoch + 1) % cfg.VALIDATE_EVERY == 0:
            val_metrics = validate(model, val_loader, cfg.DEVICE)
        else:
            val_metrics = {'val_dice': 0}
        
        # Logging
        epoch_time = time.time() - epoch_start
        lr = optimizer.param_groups[0]['lr']
        stage = train_metrics.get('stage', 1)
        
        log_parts = [
            f"[S{stage}] Epoch {epoch+1}/{cfg.TOTAL_EPOCHS}",
            f"{epoch_time:.1f}s",
            f"LR: {lr:.6f}",
            f"Loss: {train_metrics['train_loss']:.4f}",
            f"Dice: {train_metrics['train_dice_loss']:.4f}",
        ]
        
        if stage == 2:
            if train_metrics.get('train_skel_loss', 0) > 0:
                log_parts.append(f"Skel: {train_metrics['train_skel_loss']:.4f}")
            if train_metrics.get('train_cldice_loss', 0) > 0:
                log_parts.append(f"clDice: {train_metrics['train_cldice_loss']:.4f}")
        
        if val_metrics['val_dice'] > 0:
            log_parts.append(f"Val: {val_metrics['val_dice']:.4f}")
        
        if train_metrics.get('nan_batches', 0) > 0:
            log_parts.append(f"NaN: {train_metrics['nan_batches']}")
        
        print(" | ".join(log_parts))
        
        ckpt_mgr.save(model, optimizer, scheduler, scaler, epoch,
                      {**train_metrics, **val_metrics})
    
    print("\n" + "="*70)
    print(f"TRAINING COMPLETE!")
    print(f"Best Val Dice: {ckpt_mgr.best_score:.4f} @ epoch {ckpt_mgr.best_epoch + 1}")
    print("="*70)
    
    return model, ckpt_mgr

print("Ready to train V6 Two-Stage (H100 optimized)!")

In [ ]:
# =============================================================================
# CELL 9: RUN V6 TWO-STAGE TRAINING
# =============================================================================

# Configuration
cfg.RESUME_TRAINING = False
cfg.DATA_FRACTION = 1.0
cfg.VALIDATE_EVERY = 5

# Batch size for 192³ patches
# H100 80GB: batch_size = 4
# A100 40GB: batch_size = 2
# T4 16GB: batch_size = 1
cfg.BATCH_SIZE = 2

print("\n" + "="*70)
print("V6 TWO-STAGE TRAINING WITH ASPP")
print("="*70)
print(f"\nKey innovations:")
print(f"  1. Two-stage training (foundation → topology fine-tuning)")
print(f"  2. ASPP (Atrous Spatial Pyramid Pooling) for multi-scale context")
print(f"\nArchitecture:")
print(f"  - Patch size: {cfg.PATCH_SIZE}")
print(f"  - USE_ASPP: {cfg.USE_ASPP} (dilations [1,2,4,8] + GAP)")
print(f"\nStage 1 (epochs 0-{cfg.STAGE1_EPOCHS}): Build high-Dice foundation")
print(f"  - Loss: Dice ({cfg.STAGE1_DICE_WEIGHT}) + BCE ({cfg.STAGE1_BCE_WEIGHT})")
print(f"  - LR: {cfg.STAGE1_LR} with polynomial decay")
print(f"  - Goal: Val Dice 0.65+")
print(f"\nStage 2 (epochs {cfg.STAGE1_EPOCHS}-{cfg.TOTAL_EPOCHS}): Topology fine-tuning")
print(f"  - Loss: Dice ({cfg.STAGE2_DICE_WEIGHT}) + BCE ({cfg.STAGE2_BCE_WEIGHT}) + Skel ({cfg.STAGE2_SKELETON_WEIGHT}) + clDice ({cfg.STAGE2_CLDICE_WEIGHT})")
print(f"  - LR: {cfg.STAGE2_LR} (10x lower, fresh optimizer)")
print(f"  - Warmup: {cfg.SKELETON_WARMUP_EPOCHS} epochs for topology losses")
print(f"  - Goal: Maintain Val Dice 0.60+ with better topology")
print(f"\nTotal epochs: {cfg.TOTAL_EPOCHS}")
print("="*70 + "\n")

# START TRAINING
model, ckpt_mgr = main_training()